In [1]:
# =================================================================================================
# NHANES 2017-2018 DIETARY MERGE - FIDELITY PRESERVING
#
# SOURCE FILES:
#
#   DR1TOT_J_corrected.csv
#   DR2TOT_J_corrected.csv
#
# OUTPUT:
#
#   neighbourhood_nhanes_dietary_2017_2018_raw_corrected.csv
#
#
# COMPONENT:
#   Dietary
#
# FILES:
#   Dietary Interview - Total Nutrient Intakes, First Day
#   Dietary Interview - Total Nutrient Intakes, Second Day
#
#
# OFFICIAL CDC 2017-2018 STRUCTURE
# -------------------------------------------------------------------------------------------------
#
# DR1TOT_J:
#
#   Rows    = 8,704
#   Columns = 168
#
#
# DR2TOT_J:
#
#   Rows    = 8,704
#   Columns = 85
#
#
# SHARED VARIABLES
# -------------------------------------------------------------------------------------------------
#
# CDC identifies these variables as having the SAME NAME
# in Day 1 and Day 2:
#
#   SEQN
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#
#
# Strategy:
#
#   1. Validate the two files contain the same participant SEQN set.
#
#   2. Validate WTDRD1 numerically by SEQN.
#
#   3. Validate WTDR2D numerically by SEQN.
#
#   4. Validate DRABF numerically by SEQN.
#
#   5. Validate DRDINT numerically by SEQN.
#
#   6. Keep the Day-1 copy as the canonical copy.
#
#   7. Remove shared WTDRD1 / WTDR2D / DRABF / DRDINT
#      from DR2TOT_J before merging.
#
#   8. Preserve ALL Day-1-specific and Day-2-specific
#      source text exactly.
#
#
# OFFICIAL CDC SAMPLE COUNTS
# -------------------------------------------------------------------------------------------------
#
# Total participants                           = 8,704
#
# WTDRD1:
#
#   positive Day-1 weight                     = 7,641
#   zero = Day-1 incomplete/not done          = 1,063
#   missing                                   = 0
#
#
# WTDR2D:
#
#   positive Day-2 weight                     = 6,639
#   zero among Day-1 participants             = 1,002
#   missing                                   = 1,063
#
#
# DRDINT:
#
#   1 = Day 1 only                            = 1,002
#   2 = Day 1 and Day 2                       = 6,639
#   missing                                   = 1,063
#
#
# DRABF:
#
#   1 = Yes                                   = 159
#   2 = No                                    = 7,482
#   missing                                   = 1,063
#
#
# FINAL EXPECTED MERGED STRUCTURE
# -------------------------------------------------------------------------------------------------
#
# Day 1 columns      = 168
#
# Day 2 columns      = 85
#
# Shared fields      = 5 including SEQN
#
# Day 2 unique cols  = 80
#
# Final columns:
#
#       168 + 80 = 248
#
# Final rows:
#
#       8,704
#
#
# FIDELITY RULES
# -------------------------------------------------------------------------------------------------
#
# - Read corrected CSVs with:
#
#       dtype=str
#       keep_default_na=False
#
# - Dietary decimals remain AS IS.
#
# - Never call round().
#
# - Never use float_format=.
#
# - Never convert nutrient variables back to float for storage.
#
# - Numeric parsing is allowed ONLY for validation.
#
# - SEQN remains integer-looking text.
#
# - Preserve long floating representations exactly, for example:
#
#       0.15100000000000002
#       116.50000000000001
#
# - Do not cosmetically shorten dietary decimals.
#
# - No _X / _Y merge-generated fields.
#
# - No engineered Day-2 duplicate variables.
#
# - No known SAS XPORT tiny-zero artifact.
#
# - Exact source -> merged text fidelity.
#
# - Exact pre-save -> post-save text fidelity.
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DIETARY_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Dietary"
)


DR1_FILE = (
    DIETARY_DIR
    / "DR1TOT_J_corrected.csv"
)


DR2_FILE = (
    DIETARY_DIR
    / "DR2TOT_J_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2017_2018"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2017_2018_raw_corrected.csv"
)


# =================================================================================================
# 3. VERIFY SOURCE FILES EXIST
# =================================================================================================

FILES = {

    "DR1TOT_J": DR1_FILE,

    "DR2TOT_J": DR2_FILE

}


for name, path in FILES.items():

    if not path.exists():

        raise FileNotFoundError(
            f"{name} corrected file not found:\n{path}"
        )


print(
    "PASS: Both 2017-2018 Dietary corrected files exist."
)


# =================================================================================================
# 4. READ CORRECTED CSV FILES AS TEXT
# =================================================================================================

#
# CRITICAL FIDELITY RULE:
#
# DO NOT allow pandas to reinterpret corrected decimal text.
#

day1 = pd.read_csv(

    DR1_FILE,

    dtype=str,

    keep_default_na=False

)


day2 = pd.read_csv(

    DR2_FILE,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 5. STANDARDIZE COLUMN NAMES ONLY
# =================================================================================================

#
# This does NOT modify data values.
#

day1.columns = (

    day1.columns

    .str.strip()

    .str.upper()

)


day2.columns = (

    day2.columns

    .str.strip()

    .str.upper()

)


# =================================================================================================
# 6. SOURCE STRUCTURE SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2017-2018 DIETARY MERGE"
)


print(
    "=" * 110
)


print(
    "DR1TOT_J:"
)


print(
    "  Rows:",
    f"{len(day1):,}"
)


print(
    "  Columns:",
    len(day1.columns)
)


print(
    ""
)


print(
    "DR2TOT_J:"
)


print(
    "  Rows:",
    f"{len(day2):,}"
)


print(
    "  Columns:",
    len(day2.columns)
)


# =================================================================================================
# 7. OFFICIAL SOURCE STRUCTURE CHECK
# =================================================================================================

EXPECTED_ROWS = 8704

EXPECTED_DR1_COLUMNS = 168

EXPECTED_DR2_COLUMNS = 85


if len(day1) != EXPECTED_ROWS:

    raise ValueError(
        "DR1TOT_J row count differs.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day1):,}"
    )


if len(day2) != EXPECTED_ROWS:

    raise ValueError(
        "DR2TOT_J row count differs.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day2):,}"
    )


if len(day1.columns) != EXPECTED_DR1_COLUMNS:

    raise ValueError(
        "DR1TOT_J column count differs.\n"
        f"Expected: {EXPECTED_DR1_COLUMNS}\n"
        f"Actual:   {len(day1.columns)}"
    )


if len(day2.columns) != EXPECTED_DR2_COLUMNS:

    raise ValueError(
        "DR2TOT_J column count differs.\n"
        f"Expected: {EXPECTED_DR2_COLUMNS}\n"
        f"Actual:   {len(day2.columns)}"
    )


print(
    "PASS: Source dimensions match 2017-2018 expectations."
)


# =================================================================================================
# 8. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataset,
    dataset_name
):

    print(
        f"\n--- SEQN CHECK: {dataset_name} ---"
    )


    if "SEQN" not in dataset.columns:

        raise ValueError(
            f"{dataset_name}: SEQN missing."
        )


    blank_count = int(

        dataset["SEQN"]
        .eq("")
        .sum()

    )


    duplicate_count = int(

        dataset["SEQN"]
        .duplicated()
        .sum()

    )


    dot_zero_count = int(

        (
            dataset["SEQN"].ne("")
            &
            dataset["SEQN"]
            .str.endswith(".0")
        )

        .sum()

    )


    invalid_format_count = int(

        (
            dataset["SEQN"].ne("")
            &
            ~dataset["SEQN"]
            .str.fullmatch(
                r"\d+"
            )
        )

        .sum()

    )


    print(
        "Rows:",
        f"{len(dataset):,}"
    )


    print(
        "Unique SEQN:",
        f"{dataset['SEQN'].nunique():,}"
    )


    print(
        "Blank:",
        blank_count
    )


    print(
        "Duplicate:",
        duplicate_count
    )


    print(
        "Ending '.0':",
        dot_zero_count
    )


    print(
        "Invalid format:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{dataset_name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{dataset_name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{dataset_name}: SEQN contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{dataset_name}: invalid SEQN format."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 9. VALIDATE SEQN
# =================================================================================================

validate_seqn(
    day1,
    "DR1TOT_J"
)


validate_seqn(
    day2,
    "DR2TOT_J"
)


# =================================================================================================
# 10. VERIFY SAME SEQN SET
# =================================================================================================

day1_seqn = set(
    day1["SEQN"]
)


day2_seqn = set(
    day2["SEQN"]
)


only_day1 = (
    day1_seqn
    -
    day2_seqn
)


only_day2 = (
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- DAY 1 / DAY 2 SEQN SET CHECK ---"
)


print(
    "Only Day 1:",
    len(only_day1)
)


print(
    "Only Day 2:",
    len(only_day2)
)


if only_day1:

    raise ValueError(
        "SEQN values exist only in DR1TOT_J:\n"
        + str(
            sorted(
                only_day1
            )[:20]
        )
    )


if only_day2:

    raise ValueError(
        "SEQN values exist only in DR2TOT_J:\n"
        + str(
            sorted(
                only_day2
            )[:20]
        )
    )


if len(day1_seqn) != 8704:

    raise ValueError(
        "Unexpected unified Dietary SEQN set size."
    )


print(
    "PASS: DR1TOT_J and DR2TOT_J contain "
    "the same 8,704 participants."
)


# =================================================================================================
# 11. IDENTIFY ALL CROSS-FILE OVERLAPS
# =================================================================================================

overlap_columns = sorted(

    set(
        day1.columns
    )

    &

    set(
        day2.columns
    )

)


print(
    "\n--- CROSS-FILE OVERLAPS ---"
)


print(
    overlap_columns
)


EXPECTED_SHARED_COLUMNS = [

    "DRABF",

    "DRDINT",

    "SEQN",

    "WTDR2D",

    "WTDRD1"

]


if overlap_columns != EXPECTED_SHARED_COLUMNS:

    raise ValueError(
        "Unexpected DR1TOT_J / DR2TOT_J shared-variable set.\n\n"
        f"Expected:\n{EXPECTED_SHARED_COLUMNS}\n\n"
        f"Actual:\n{overlap_columns}"
    )


print(
    "PASS: Exactly five official shared variables found."
)


# =================================================================================================
# 12. NUMERIC SHARED-VARIABLE COMPARISON FUNCTION
# =================================================================================================

#
# Data remain STRINGS.
#
# Numeric parsing occurs ONLY on temporary copies for validation.
#
# This means:
#
#     "0"
#
# and:
#
#     "0.0"
#
# are scientifically equal.
#
# But the canonical Day-1 source text is retained exactly.
#

def compare_numeric_variable_by_seqn(
    left_df,
    right_df,
    column,
    left_name,
    right_name,
    require_missing_match=True
):

    print(
        "\n"
        + "-" * 105
    )


    print(
        f"SHARED VARIABLE CHECK: "
        f"{left_name}.{column} "
        f"<-> "
        f"{right_name}.{column}"
    )


    print(
        "-" * 105
    )


    if column not in left_df.columns:

        raise ValueError(
            f"{column} missing from {left_name}."
        )


    if column not in right_df.columns:

        raise ValueError(
            f"{column} missing from {right_name}."
        )


    left_part = (

        left_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}__LEFT"
            }
        )

    )


    right_part = (

        right_df[
            [
                "SEQN",
                column
            ]
        ]

        .rename(
            columns={
                column:
                    f"{column}__RIGHT"
            }
        )

    )


    comparison = (

        left_part

        .merge(

            right_part,

            on="SEQN",

            how="outer",

            validate="one_to_one",

            indicator=True

        )

        .fillna("")

    )


    if not comparison["_merge"].eq("both").all():

        raise ValueError(
            f"{column}: Day 1 and Day 2 do not "
            "contain identical participant sets."
        )


    left_raw = (
        comparison[
            f"{column}__LEFT"
        ]
    )


    right_raw = (
        comparison[
            f"{column}__RIGHT"
        ]
    )


    left_populated = (
        left_raw.ne("")
    )


    right_populated = (
        right_raw.ne("")
    )


    missing_mismatch_count = int(

        (
            left_populated
            !=
            right_populated
        )

        .sum()

    )


    print(
        "Missing-position mismatches:",
        f"{missing_mismatch_count:,}"
    )


    if (
        require_missing_match
        and
        missing_mismatch_count != 0
    ):

        raise ValueError(
            f"{column}: missing positions differ "
            "between DR1TOT_J and DR2TOT_J."
        )


    both_populated = (

        left_populated

        &

        right_populated

    )


    both_count = int(
        both_populated.sum()
    )


    left_numeric = pd.to_numeric(

        left_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    right_numeric = pd.to_numeric(

        right_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        left_numeric.to_numpy(
            dtype=float
        ),

        right_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(

        (
            ~numeric_equal
        )

        .sum()

    )


    raw_text_difference_count = int(

        (
            left_raw.loc[
                both_populated
            ]
            .reset_index(
                drop=True
            )

            !=

            right_raw.loc[
                both_populated
            ]
            .reset_index(
                drop=True
            )
        )

        .sum()

    )


    print(
        "Both populated:",
        f"{both_count:,}"
    )


    print(
        "Raw text-format differences:",
        f"{raw_text_difference_count:,}"
    )


    print(
        "TRUE numeric mismatches:",
        f"{numeric_mismatch_count:,}"
    )


    if numeric_mismatch_count != 0:

        raise ValueError(
            f"{column}: true numeric differences exist "
            "between DR1TOT_J and DR2TOT_J."
        )


    print(
        "PASS: Shared values agree numerically."
    )


    return {

        "both_populated":
            both_count,

        "missing_mismatches":
            missing_mismatch_count,

        "text_differences":
            raw_text_difference_count,

        "numeric_mismatches":
            numeric_mismatch_count

    }


# =================================================================================================
# 13. VALIDATE WTDRD1 BETWEEN DAY 1 AND DAY 2
# =================================================================================================

wtdrd1_result = (

    compare_numeric_variable_by_seqn(

        day1,

        day2,

        "WTDRD1",

        "DR1TOT_J",

        "DR2TOT_J"

    )

)


# =================================================================================================
# 14. VALIDATE WTDR2D BETWEEN DAY 1 AND DAY 2
# =================================================================================================

wtdr2d_result = (

    compare_numeric_variable_by_seqn(

        day1,

        day2,

        "WTDR2D",

        "DR1TOT_J",

        "DR2TOT_J"

    )

)


# =================================================================================================
# 15. VALIDATE DRABF BETWEEN DAY 1 AND DAY 2
# =================================================================================================

drabf_result = (

    compare_numeric_variable_by_seqn(

        day1,

        day2,

        "DRABF",

        "DR1TOT_J",

        "DR2TOT_J"

    )

)


# =================================================================================================
# 16. VALIDATE DRDINT BETWEEN DAY 1 AND DAY 2
# =================================================================================================

drdint_result = (

    compare_numeric_variable_by_seqn(

        day1,

        day2,

        "DRDINT",

        "DR1TOT_J",

        "DR2TOT_J"

    )

)


print(
    "\nPASS: All four duplicated shared data variables "
    "agree between DR1TOT_J and DR2TOT_J."
)


print(
    "Canonical source for shared variables: DR1TOT_J"
)


# =================================================================================================
# 17. CDC WTDRD1 CHECK
# =================================================================================================

wtdrd1 = pd.to_numeric(

    day1["WTDRD1"]
    .where(
        day1["WTDRD1"] != "",
        np.nan
    ),

    errors="raise"

)


wtdrd1_positive = int(

    (
        wtdrd1 > 0
    )

    .sum()

)


wtdrd1_zero = int(

    wtdrd1
    .eq(0)
    .sum()

)


wtdrd1_missing = int(

    wtdrd1
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: WTDRD1 ---"
)


print(
    "Positive:",
    f"{wtdrd1_positive:,}"
)


print(
    "Zero:",
    f"{wtdrd1_zero:,}"
)


print(
    "Missing:",
    f"{wtdrd1_missing:,}"
)


if wtdrd1_positive != 7641:

    raise ValueError(
        "WTDRD1 positive count differs from CDC."
    )


if wtdrd1_zero != 1063:

    raise ValueError(
        "WTDRD1 zero count differs from CDC."
    )


if wtdrd1_missing != 0:

    raise ValueError(
        "WTDRD1 missing count differs from CDC."
    )


print(
    "PASS: WTDRD1 matches CDC."
)


# =================================================================================================
# 18. CDC WTDR2D CHECK
# =================================================================================================

wtdr2d = pd.to_numeric(

    day1["WTDR2D"]
    .where(
        day1["WTDR2D"] != "",
        np.nan
    ),

    errors="raise"

)


wtdr2d_positive = int(

    (
        wtdr2d > 0
    )

    .sum()

)


wtdr2d_zero = int(

    wtdr2d
    .eq(0)
    .sum()

)


wtdr2d_missing = int(

    wtdr2d
    .isna()
    .sum()

)


print(
    "\n--- CDC CHECK: WTDR2D ---"
)


print(
    "Positive:",
    f"{wtdr2d_positive:,}"
)


print(
    "Zero:",
    f"{wtdr2d_zero:,}"
)


print(
    "Missing:",
    f"{wtdr2d_missing:,}"
)


if wtdr2d_positive != 6639:

    raise ValueError(
        "WTDR2D positive count differs from CDC."
    )


if wtdr2d_zero != 1002:

    raise ValueError(
        "WTDR2D zero count differs from CDC."
    )


if wtdr2d_missing != 1063:

    raise ValueError(
        "WTDR2D missing count differs from CDC."
    )


print(
    "PASS: WTDR2D matches CDC."
)


# =================================================================================================
# 19. CDC DRDINT CHECK
# =================================================================================================

drdint = (

    day1["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


EXPECTED_DRDINT = {

    "1": 1002,

    "2": 6639,

    "": 1063

}


print(
    "\n--- CDC CHECK: DRDINT ---"
)


print(
    drdint
)


if drdint != EXPECTED_DRDINT:

    raise ValueError(
        "DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{drdint}"
    )


print(
    "PASS: DRDINT matches CDC."
)


# =================================================================================================
# 20. CDC DRABF CHECK
# =================================================================================================

drabf = (

    day1["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


EXPECTED_DRABF = {

    "1": 159,

    "2": 7482,

    "": 1063

}


print(
    "\n--- CDC CHECK: DRABF ---"
)


print(
    drabf
)


if drabf != EXPECTED_DRABF:

    raise ValueError(
        "DRABF distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{drabf}"
    )


print(
    "PASS: DRABF matches CDC."
)


# =================================================================================================
# 21. VERIFY WEIGHT / DRDINT ROUTING
# =================================================================================================

#
# Scientific consistency:
#
# DRDINT=1:
#
#   completed Day 1 only
#   WTDRD1 > 0
#   WTDR2D = 0
#
#
# DRDINT=2:
#
#   completed Day 1 and Day 2
#   WTDRD1 > 0
#   WTDR2D > 0
#
#
# DRDINT missing:
#
#   WTDRD1 = 0
#   WTDR2D missing
#

drdint_text = (
    day1["DRDINT"]
)


route_day1_only = (

    drdint_text.eq("1")
    &
    (wtdrd1 > 0)
    &
    wtdr2d.eq(0)

)


route_both_days = (

    drdint_text.eq("2")
    &
    (wtdrd1 > 0)
    &
    (wtdr2d > 0)

)


route_no_complete_day1 = (

    drdint_text.eq("")
    &
    wtdrd1.eq(0)
    &
    wtdr2d.isna()

)


routing_ok = (

    route_day1_only

    |

    route_both_days

    |

    route_no_complete_day1

)


routing_mismatch_count = int(

    (
        ~routing_ok
    )

    .sum()

)


print(
    "\n--- DIETARY ROUTING CHECK ---"
)


print(
    "Day 1 only:",
    f"{route_day1_only.sum():,}"
)


print(
    "Both days:",
    f"{route_both_days.sum():,}"
)


print(
    "No complete Day 1:",
    f"{route_no_complete_day1.sum():,}"
)


print(
    "Routing mismatches:",
    routing_mismatch_count
)


if routing_mismatch_count != 0:

    raise ValueError(
        "WTDRD1 / WTDR2D / DRDINT routing mismatch."
    )


print(
    "PASS: Dietary weight/intake routing is internally consistent."
)


# =================================================================================================
# 22. CHECK KNOWN XPORT TINY TOKEN IN BOTH CORRECTED SOURCES
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


def find_known_tiny_token(
    dataset,
    dataset_name
):

    findings = {}


    for col in dataset.columns:

        count = int(

            dataset[col]

            .str.lower()

            .eq(
                TINY_TOKEN.lower()
            )

            .sum()

        )


        if count > 0:

            findings[
                col
            ] = count


    total = sum(
        findings.values()
    )


    print(
        f"\nKnown tiny XPORT token in {dataset_name}:",
        f"{total:,}"
    )


    if total != 0:

        raise ValueError(
            f"{dataset_name} still contains the known "
            "SAS XPORT tiny-zero artifact:\n"
            + str(
                findings
            )
        )


    print(
        "PASS"
    )


find_known_tiny_token(
    day1,
    "DR1TOT_J"
)


find_known_tiny_token(
    day2,
    "DR2TOT_J"
)


# =================================================================================================
# 23. DIETARY DECIMAL TEXT DIAGNOSTIC
# =================================================================================================

#
# Diagnostic only.
#
# Nothing is converted or rounded.
#

day1_nutrient_fields = [

    col

    for col in day1.columns

    if col.startswith(
        "DR1T"
    )

]


day2_nutrient_fields = [

    col

    for col in day2.columns

    if col.startswith(
        "DR2T"
    )

]


print(
    "\n"
    + "=" * 110
)


print(
    "DIETARY DECIMAL-PRESERVATION DIAGNOSTIC"
)


print(
    "=" * 110
)


print(
    "Day 1 DR1T* fields:",
    len(day1_nutrient_fields)
)


print(
    "Day 2 DR2T* fields:",
    len(day2_nutrient_fields)
)


def count_decimal_tokens(
    dataset,
    columns
):

    total = 0


    for col in columns:

        total += int(

            (
                dataset[col].ne("")
                &
                dataset[col]
                .str.contains(
                    ".",
                    regex=False
                )
            )

            .sum()

        )


    return total


day1_decimal_tokens = count_decimal_tokens(

    day1,

    day1_nutrient_fields

)


day2_decimal_tokens = count_decimal_tokens(

    day2,

    day2_nutrient_fields

)


print(
    "Day 1 nutrient decimal tokens:",
    f"{day1_decimal_tokens:,}"
)


print(
    "Day 2 nutrient decimal tokens:",
    f"{day2_decimal_tokens:,}"
)


print(
    "PASS: Source dietary values remain raw text."
)


# =================================================================================================
# 24. PREPARE DAY 2 FOR MERGE
# =================================================================================================

#
# Keep canonical Day-1 versions of:
#
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#
#
# SEQN remains as merge key.
#

DAY2_REDUNDANT_SHARED = [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


for col in DAY2_REDUNDANT_SHARED:

    if col not in day2.columns:

        raise ValueError(
            f"Expected shared Day-2 field missing: {col}"
        )


day2_for_merge = (

    day2.drop(
        columns=DAY2_REDUNDANT_SHARED
    )

)


print(
    "\nRemoved validated redundant Day-2 shared fields:"
)


for col in DAY2_REDUNDANT_SHARED:

    print(
        " -",
        col
    )


# =================================================================================================
# 25. CHECK FOR UNEXPECTED OVERLAPS BEFORE MERGE
# =================================================================================================

remaining_overlap = sorted(

    (
        set(
            day1.columns
        )

        &

        set(
            day2_for_merge.columns
        )
    )

    -

    {
        "SEQN"
    }

)


if remaining_overlap:

    raise ValueError(
        "Unexpected variable overlaps remain before merge:\n"
        + str(
            remaining_overlap
        )
    )


print(
    "PASS: No unexpected overlap remains before merge."
)


# =================================================================================================
# 26. EXPECTED DAY-2 UNIQUE FIELD COUNT
# =================================================================================================

day2_unique_count = (

    len(
        day2_for_merge.columns
    )

    - 1

)


print(
    "\nDay-2 unique fields:",
    day2_unique_count
)


if day2_unique_count != 80:

    raise ValueError(
        "Unexpected Day-2 unique field count.\n"
        f"Expected: 80\n"
        f"Actual:   {day2_unique_count}"
    )


print(
    "PASS: DR2TOT_J contributes exactly 80 unique fields."
)


# =================================================================================================
# 27. MERGE DAY 1 + DAY 2
# =================================================================================================

df = (

    day1

    .merge(

        day2_for_merge,

        on="SEQN",

        how="outer",

        validate="one_to_one"

    )

    .fillna("")

)


print(
    "\nMerged DR1TOT_J + DR2TOT_J."
)


# =================================================================================================
# 28. FINAL STRUCTURE CHECK
# =================================================================================================

EXPECTED_FINAL_ROWS = 8704

EXPECTED_FINAL_COLUMNS = 248


print(
    "\n--- FINAL STRUCTURE ---"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Unique SEQN:",
    f"{df['SEQN'].nunique():,}"
)


if len(df) != EXPECTED_FINAL_ROWS:

    raise ValueError(
        "Final Dietary row count differs.\n"
        f"Expected: {EXPECTED_FINAL_ROWS:,}\n"
        f"Actual:   {len(df):,}"
    )


if len(df.columns) != EXPECTED_FINAL_COLUMNS:

    raise ValueError(
        "Final Dietary column count differs.\n"
        f"Expected: {EXPECTED_FINAL_COLUMNS}\n"
        f"Actual:   {len(df.columns)}"
    )


if df["SEQN"].nunique() != EXPECTED_FINAL_ROWS:

    raise ValueError(
        "Final Dietary unique SEQN differs."
    )


print(
    "PASS: Final structure = 8,704 rows x 248 columns."
)


# =================================================================================================
# 29. FINAL SEQN CHECK
# =================================================================================================

if (
    df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final Dietary data contains blank SEQN."
    )


if (
    df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final Dietary data contains duplicate SEQN."
    )


if (
    df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final Dietary data contains SEQN '.0'."
    )


if (

    ~df["SEQN"]
    .str.fullmatch(
        r"\d+"
    )

).any():

    raise ValueError(
        "Final Dietary data contains invalid SEQN."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 30. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_columns = (

    df.columns[
        df.columns.duplicated()
    ]

    .tolist()

)


if duplicate_columns:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_columns
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 31. VERIFY EXACTLY ONE COPY OF EACH SHARED VARIABLE
# =================================================================================================

for col in [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]:

    matching = [

        name

        for name in df.columns

        if name == col

    ]


    if len(matching) != 1:

        raise ValueError(
            f"Expected exactly one final {col} field."
        )


print(
    "PASS: One canonical copy of each shared variable retained."
)


# =================================================================================================
# 32. NO MERGE-GENERATED SUFFIXES
# =================================================================================================

BAD_SUFFIXES = [

    "_X",

    "_Y",

    "_DAY1",

    "_DAY2"

]


bad_suffix_fields = []


for col in df.columns:

    if any(

        col.endswith(
            suffix
        )

        for suffix in BAD_SUFFIXES

    ):

        bad_suffix_fields.append(
            col
        )


if bad_suffix_fields:

    raise ValueError(
        "Unexpected merge-generated suffix fields:\n"
        + str(
            bad_suffix_fields
        )
    )


print(
    "PASS: No uncontrolled merge suffix fields."
)


# =================================================================================================
# 33. OLD ENGINEERED DUPLICATE FIELD CHECK
# =================================================================================================

#
# These were problematic in older unified files.
#
# They must NOT be created.
#

BAD_ENGINEERED_FIELDS = [

    "WTDRD1_DAY_2_TOTAL_INTAKE",

    "WTDR2D_DAY_2_TOTAL_INTAKE",

    "DRABF_DAY_2_TOTAL_INTAKE",

    "DRDINT_DAY_2_TOTAL_INTAKE"

]


bad_engineered_found = [

    col

    for col in BAD_ENGINEERED_FIELDS

    if col in df.columns

]


if bad_engineered_found:

    raise ValueError(
        "Engineered dietary duplicate fields remain:\n"
        + str(
            bad_engineered_found
        )
    )


print(
    "PASS: No engineered Day-2 duplicate fields."
)


# =================================================================================================
# 34. SOURCE TEXT FIDELITY FUNCTION
# =================================================================================================

def validate_source_text_fidelity(
    source_df,
    final_df,
    source_name,
    retained_columns
):

    print(
        f"\n--- SOURCE TEXT FIDELITY: {source_name} ---"
    )


    source_check = (

        source_df[
            ["SEQN"]
            +
            retained_columns
        ]

        .copy()

    )


    final_check = (

        final_df[
            ["SEQN"]
            +
            retained_columns
        ]

        .copy()

    )


    final_check = (

        final_check.rename(

            columns={

                col:
                    f"{col}__FINAL"

                for col in retained_columns

            }

        )

    )


    check = (

        source_check

        .merge(

            final_check,

            on="SEQN",

            how="left",

            validate="one_to_one"

        )

        .fillna("")

    )


    failures = []


    for col in retained_columns:

        mismatch_count = int(

            (
                check[col]
                !=
                check[
                    f"{col}__FINAL"
                ]
            )

            .sum()

        )


        if mismatch_count > 0:

            failures.append(
                (
                    col,
                    mismatch_count
                )
            )


    if failures:

        raise ValueError(
            f"{source_name} retained-field fidelity failures:\n"
            + str(
                failures
            )
        )


    print(
        "PASS:",
        len(retained_columns),
        "fields match source text-for-text."
    )


# =================================================================================================
# 35. DAY-1 SOURCE TEXT FIDELITY
# =================================================================================================

DAY1_RETAINED = [

    col

    for col in day1.columns

    if col != "SEQN"

]


validate_source_text_fidelity(

    day1,

    df,

    "DR1TOT_J",

    DAY1_RETAINED

)


# =================================================================================================
# 36. DAY-2 SOURCE TEXT FIDELITY
# =================================================================================================

#
# Exclude intentionally deduplicated fields.
#

DAY2_RETAINED = [

    col

    for col in day2.columns

    if col not in {

        "SEQN",

        "WTDRD1",

        "WTDR2D",

        "DRABF",

        "DRDINT"

    }

]


validate_source_text_fidelity(

    day2,

    df,

    "DR2TOT_J",

    DAY2_RETAINED

)


# =================================================================================================
# 37. FINAL WTDRD1 CHECK
# =================================================================================================

final_wtdrd1 = pd.to_numeric(

    df["WTDRD1"],

    errors="raise"

)


if int(
    (final_wtdrd1 > 0).sum()
) != 7641:

    raise ValueError(
        "Final WTDRD1 positive count differs."
    )


if int(
    final_wtdrd1.eq(0).sum()
) != 1063:

    raise ValueError(
        "Final WTDRD1 zero count differs."
    )


print(
    "PASS: Final WTDRD1 = 7,641 positive + 1,063 zero."
)


# =================================================================================================
# 38. FINAL WTDR2D CHECK
# =================================================================================================

final_wtdr2d = pd.to_numeric(

    df["WTDR2D"]
    .where(
        df["WTDR2D"] != "",
        np.nan
    ),

    errors="raise"

)


if int(
    (final_wtdr2d > 0).sum()
) != 6639:

    raise ValueError(
        "Final WTDR2D positive count differs."
    )


if int(
    final_wtdr2d.eq(0).sum()
) != 1002:

    raise ValueError(
        "Final WTDR2D zero count differs."
    )


if int(
    final_wtdr2d.isna().sum()
) != 1063:

    raise ValueError(
        "Final WTDR2D missing count differs."
    )


print(
    "PASS: Final WTDR2D = "
    "6,639 positive + 1,002 zero + 1,063 missing."
)


# =================================================================================================
# 39. FINAL DRDINT CHECK
# =================================================================================================

final_drdint = (

    df["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


if final_drdint != EXPECTED_DRDINT:

    raise ValueError(
        "Final DRDINT distribution differs.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{final_drdint}"
    )


print(
    "PASS: Final DRDINT distribution."
)


# =================================================================================================
# 40. FINAL DRABF CHECK
# =================================================================================================

final_drabf = (

    df["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


if final_drabf != EXPECTED_DRABF:

    raise ValueError(
        "Final DRABF distribution differs.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{final_drabf}"
    )


print(
    "PASS: Final DRABF distribution."
)


# =================================================================================================
# 41. FINAL KNOWN XPORT TINY-ARTIFACT CHECK
# =================================================================================================

tiny_artifact_columns = {}


for col in df.columns:

    count = int(

        df[col]

        .str.lower()

        .eq(
            TINY_TOKEN.lower()
        )

        .sum()

    )


    if count > 0:

        tiny_artifact_columns[
            col
        ] = count


tiny_artifact_count = sum(
    tiny_artifact_columns.values()
)


print(
    "\nKnown XPORT tiny artifacts:",
    f"{tiny_artifact_count:,}"
)


if tiny_artifact_count != 0:

    raise ValueError(
        "Known XPORT tiny-zero artifacts remain:\n"
        + str(
            tiny_artifact_columns
        )
    )


print(
    "PASS: No known XPORT tiny-zero artifacts."
)


# =================================================================================================
# 42. REPRESENTATIVE DIETARY DECIMAL FIDELITY
# =================================================================================================

#
# NO numeric conversion here.
#
# We merely display source text examples.
#

REPRESENTATIVE_DIETARY_FIELDS = [

    "WTDRD1",

    "WTDR2D",

    "DR1TPROT",

    "DR1TCARB",

    "DR1TTFAT",

    "DR1TP204",

    "DR1TP205",

    "DR1TP226",

    "DR2TPROT",

    "DR2TCARB",

    "DR2TTFAT",

    "DR2TP204",

    "DR2TP205",

    "DR2TP226"

]


print(
    "\n"
    + "=" * 110
)


print(
    "REPRESENTATIVE DIETARY DECIMAL FIDELITY"
)


print(
    "=" * 110
)


for col in REPRESENTATIVE_DIETARY_FIELDS:

    if col not in df.columns:

        continue


    populated = (

        df.loc[
            df[col].ne(""),
            col
        ]

    )


    decimal_count = int(

        populated

        .str.contains(
            ".",
            regex=False
        )

        .sum()

    )


    print(
        "\n",
        col,
        sep=""
    )


    print(
        "  populated:",
        f"{len(populated):,}"
    )


    print(
        "  tokens containing decimal point:",
        f"{decimal_count:,}"
    )


    print(
        "  examples:",
        populated.head(5).tolist()
    )


print(
    "\nPASS: Dietary values remain exact corrected-source text."
)


# =================================================================================================
# 43. SAVE FINAL DIETARY CSV
# =================================================================================================

#
# CRITICAL:
#
# DO NOT USE:
#
#     float_format=
#
# DO NOT:
#
#     round()
#
# DO NOT:
#
#     astype(float)
#
# before saving.
#

df.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved final Dietary CSV:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 44. RE-READ FINAL CSV AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 45. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != df.shape:

    raise ValueError(
        "CSV shape changed after save.\n"
        f"Before: {df.shape}\n"
        f"After:  {final_df.shape}"
    )


if (
    final_df.columns.tolist()
    !=
    df.columns.tolist()
):

    raise ValueError(
        "CSV column structure/order changed after save."
    )


print(
    "PASS: CSV structure preserved."
)


# =================================================================================================
# 46. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

post_save_failures = []


for col in df.columns:

    mismatch_count = int(

        (
            df[col]
            !=
            final_df[col]
        )

        .sum()

    )


    if mismatch_count > 0:

        post_save_failures.append(
            (
                col,
                mismatch_count
            )
        )


if post_save_failures:

    raise ValueError(
        "Dietary text changed during CSV export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 47. FINAL SAVED SEQN CHECK
# =================================================================================================

if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Saved CSV contains blank SEQN."
    )


if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Saved CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Saved CSV contains SEQN '.0'."
    )


if (

    ~final_df["SEQN"]
    .str.fullmatch(
        r"\d+"
    )

).any():

    raise ValueError(
        "Saved CSV contains invalid SEQN."
    )


print(
    "PASS: Saved CSV SEQN integrity."
)


# =================================================================================================
# 48. FINAL DUPLICATE FIELD CHECK
# =================================================================================================

duplicate_column_names = (

    final_df.columns[
        final_df.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Saved CSV has duplicate column names:\n"
        + str(
            duplicate_column_names
        )
    )


bad_engineered_found = [

    col

    for col in BAD_ENGINEERED_FIELDS

    if col in final_df.columns

]


if bad_engineered_found:

    raise ValueError(
        "Saved CSV contains engineered duplicate fields:\n"
        + str(
            bad_engineered_found
        )
    )


print(
    "PASS: No duplicate/engineered Dietary fields."
)


# =================================================================================================
# 49. FINAL SHARED FIELD COUNT CHECK
# =================================================================================================

for col in [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]:

    count = int(

        sum(
            final_df.columns
            == col
        )

    )


    if count != 1:

        raise ValueError(
            f"Saved file should contain exactly one {col}."
        )


print(
    "PASS: Exactly one canonical shared-variable set."
)


# =================================================================================================
# 50. FINAL PREFIX CHECK
# =================================================================================================

final_dr1_fields = [

    col

    for col in final_df.columns

    if col.startswith(
        "DR1"
    )

]


final_dr2_fields = [

    col

    for col in final_df.columns

    if col.startswith(
        "DR2"
    )

]


print(
    "\n--- FINAL DAY PREFIX COUNTS ---"
)


print(
    "DR1* fields:",
    len(final_dr1_fields)
)


print(
    "DR2* fields:",
    len(final_dr2_fields)
)


if len(final_dr1_fields) == 0:

    raise ValueError(
        "No Day-1 Dietary fields found."
    )


if len(final_dr2_fields) == 0:

    raise ValueError(
        "No Day-2 Dietary fields found."
    )


print(
    "PASS: Both Day-1 and Day-2 fields retained."
)


# =================================================================================================
# 51. FINAL CDC RECHECK
# =================================================================================================

saved_wtdrd1 = pd.to_numeric(

    final_df["WTDRD1"],

    errors="raise"

)


saved_wtdr2d = pd.to_numeric(

    final_df["WTDR2D"]
    .where(
        final_df["WTDR2D"] != "",
        np.nan
    ),

    errors="raise"

)


if int(
    (saved_wtdrd1 > 0).sum()
) != 7641:

    raise ValueError(
        "Saved WTDRD1 positive count differs."
    )


if int(
    saved_wtdrd1.eq(0).sum()
) != 1063:

    raise ValueError(
        "Saved WTDRD1 zero count differs."
    )


if int(
    (saved_wtdr2d > 0).sum()
) != 6639:

    raise ValueError(
        "Saved WTDR2D positive count differs."
    )


if int(
    saved_wtdr2d.eq(0).sum()
) != 1002:

    raise ValueError(
        "Saved WTDR2D zero count differs."
    )


if int(
    saved_wtdr2d.isna().sum()
) != 1063:

    raise ValueError(
        "Saved WTDR2D missing count differs."
    )


print(
    "PASS: Final saved Dietary weight benchmarks."
)


# =================================================================================================
# 52. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2017-2018 DIETARY MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2017-2018"
)


print(
    ""
)


print(
    "Component:"
)


print(
    "  Dietary"
)


print(
    ""
)


print(
    "Source files:"
)


print(
    "  DR1TOT_J_corrected.csv"
)


print(
    "  DR2TOT_J_corrected.csv"
)


print(
    ""
)


print(
    "Source rows:"
)


print(
    "  DR1TOT_J = 8,704"
)


print(
    "  DR2TOT_J = 8,704"
)


print(
    ""
)


print(
    "Source columns:"
)


print(
    "  DR1TOT_J = 168"
)


print(
    "  DR2TOT_J = 85"
)


print(
    ""
)


print(
    "Official shared variables:"
)


print(
    "  SEQN"
)


print(
    "  WTDRD1"
)


print(
    "  WTDR2D"
)


print(
    "  DRABF"
)


print(
    "  DRDINT"
)


print(
    ""
)


print(
    "Canonical shared source:"
)


print(
    "  DR1TOT_J"
)


print(
    ""
)


print(
    "Final rows:",
    f"{len(final_df):,}"
)


print(
    "Final columns:",
    len(final_df.columns)
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    ""
)


print(
    "WTDRD1:"
)


print(
    "  Positive = 7,641"
)


print(
    "  Zero     = 1,063"
)


print(
    "  Missing  = 0"
)


print(
    ""
)


print(
    "WTDR2D:"
)


print(
    "  Positive = 6,639"
)


print(
    "  Zero     = 1,002"
)


print(
    "  Missing  = 1,063"
)


print(
    ""
)


print(
    "DRDINT:"
)


print(
    "  Day 1 only    = 1,002"
)


print(
    "  Day 1 + Day 2 = 6,639"
)


print(
    "  Missing       = 1,063"
)


print(
    ""
)


print(
    "DRABF:"
)


print(
    "  Yes     = 159"
)


print(
    "  No      = 7,482"
)


print(
    "  Missing = 1,063"
)


print(
    ""
)


print(
    "Shared-variable numeric mismatches:"
)


print(
    "  WTDRD1 =",
    wtdrd1_result[
        "numeric_mismatches"
    ]
)


print(
    "  WTDR2D =",
    wtdr2d_result[
        "numeric_mismatches"
    ]
)


print(
    "  DRABF  =",
    drabf_result[
        "numeric_mismatches"
    ]
)


print(
    "  DRDINT =",
    drdint_result[
        "numeric_mismatches"
    ]
)


print(
    ""
)


print(
    "SEQN '.0': 0"
)


print(
    "Duplicate SEQN: 0"
)


print(
    "Duplicate column names: 0"
)


print(
    "Engineered Day-2 duplicate fields: 0"
)


print(
    "Known XPORT tiny artifacts: 0"
)


print(
    "Dietary decimal rounding: 0"
)


print(
    "Dietary decimal reformatting during merge: 0"
)


print(
    "Day-1 retained-source fidelity: PASS"
)


print(
    "Day-2 retained-source fidelity: PASS"
)


print(
    "Pre-save/post-save exact text fidelity: PASS"
)


print(
    "=" * 110
)

PASS: Both 2017-2018 Dietary corrected files exist.

NHANES 2017-2018 DIETARY MERGE
DR1TOT_J:
  Rows: 8,704
  Columns: 168

DR2TOT_J:
  Rows: 8,704
  Columns: 85
PASS: Source dimensions match 2017-2018 expectations.

--- SEQN CHECK: DR1TOT_J ---
Rows: 8,704
Unique SEQN: 8,704
Blank: 0
Duplicate: 0
Ending '.0': 0
Invalid format: 0
PASS

--- SEQN CHECK: DR2TOT_J ---
Rows: 8,704
Unique SEQN: 8,704
Blank: 0
Duplicate: 0
Ending '.0': 0
Invalid format: 0
PASS

--- DAY 1 / DAY 2 SEQN SET CHECK ---
Only Day 1: 0
Only Day 2: 0
PASS: DR1TOT_J and DR2TOT_J contain the same 8,704 participants.

--- CROSS-FILE OVERLAPS ---
['DRABF', 'DRDINT', 'SEQN', 'WTDR2D', 'WTDRD1']
PASS: Exactly five official shared variables found.

---------------------------------------------------------------------------------------------------------
SHARED VARIABLE CHECK: DR1TOT_J.WTDRD1 <-> DR2TOT_J.WTDRD1
---------------------------------------------------------------------------------------------------------
Missing-po